In [ ]:
#| default_exp data_loaders

In [ ]:
#|export
import pandas as pd
import yfinance as yf
import torch
import numpy as np
from sklearn.preprocessing import StandardScaler
from torch.utils.data import Dataset, DataLoader

In [ ]:
#| export
def get_nifty_regime_data(start_date="2015-01-01", end_date="2026-03-01"):
    nifty = yf.download("^NSEI", start=start_date, end=end_date, auto_adjust=True)
    vix_data = yf.download("^INDIAVIX", start=start_date, end=end_date, auto_adjust=True)
    if isinstance(nifty.columns, pd.MultiIndex):
        df = pd.DataFrame({
            'Close': nifty['Close'].iloc[:, 0],
            'Volume': nifty['Volume'].iloc[:, 0]
        })
        vix = vix_data['Close'].iloc[:, 0]
    else:
        df = nifty[['Close', 'Volume']].copy()
        vix = vix_data['Close']

    df['log_return'] = np.log(df['Close'] / df['Close'].shift(1))
    df['skew'] = df['log_return'].rolling(60).skew().shift(1)
    df['kurtosis'] = df['log_return'].rolling(60).kurt().shift(1)
    df['realized_vol'] = df['log_return'].rolling(21).std().shift(1) * np.sqrt(252)
    df['vol_spike'] = df['Volume'] / df['Volume'].rolling(20).mean().shift(1)
    delta = df['Close'].diff()
    gain = (delta.where(delta > 0, 0)).rolling(14).mean().shift(1)
    loss = (-delta.where(delta < 0, 0)).rolling(14).mean().shift(1)
    df['RSI'] = 100 - (100 / (1 + (gain / loss)))
    df['RSI_velocity'] = df['RSI'].diff(3)
    df['sma_200'] = df['Close'].rolling(200).mean().shift(1)

    df['regime'] = 0
    df.loc[
        (df['Close'] < df['sma_200']) |
        ((df['skew'] < -0.5) & (df['realized_vol'] > df['realized_vol'].rolling(100).mean().shift(1))),
        'regime'
    ] = 1
    df['target_return'] = df['log_return'].shift(-1)
    vix_aligned = pd.Series(vix, name='VIX').reindex(df.index, method='ffill')
    df['VIX'] = vix_aligned.shift(1)
    df = df.dropna()
    cols = ['target_return', 'realized_vol', 'VIX', 'RSI', 'skew', 'kurtosis', 'vol_spike', 'RSI_velocity','sma_200','Close','regime']
    return df[cols]

In [ ]:
#| export
def compute_regime_score(df):
    score = pd.Series(0.0, index=df.index)
    sma_distance = (df['sma_200'] - df['Close'].shift(1)) / df['sma_200']
    signal_1 = sma_distance.clip(0, 0.15) / 0.15  
    vol_percentile = df['realized_vol'].rolling(252).rank(pct=True).shift(1)
    signal_2 = vol_percentile.fillna(0.5)
    skew_stress = (-df['skew']).clip(0, 2) / 2 
    signal_3 = skew_stress
    vix_percentile = df['VIX'].rolling(252).rank(pct=True).shift(1)
    signal_4 = vix_percentile.fillna(0.5)
    score = (0.30 * signal_1 +
             0.30 * signal_2 +
             0.20 * signal_3 +
             0.20 * signal_4)

    return score.clip(0, 1)



In [ ]:
risk_df = get_nifty_regime_data()
risk_df['regime_score'] = compute_regime_score(risk_df)
risk_df['regime_score'] = risk_df['regime_score'].fillna(0.5)
risk_df['regime'] = (risk_df['regime_score'] > 0.5).astype(int)
risk_df.drop(columns=['sma_200','Close'], inplace=True)
risk_df.head()

[*********************100%***********************]  1 of 1 completed

[*********************100%***********************]  1 of 1 completed



,target_return,realized_vol,VIX,RSI,skew,kurtosis,vol_spike,RSI_velocity,regime,regime_score
Date,,,,,,,,,,
2015-10-27,-0.007523,0.110922,17.230000,63.648657,-1.867438,7.837366,0.923993,-16.027928,0,0.500000
2015-10-28,-0.007302,0.112642,16.540001,57.820192,-1.840251,7.784645,1.129109,-17.032109,0,0.467612
2015-10-29,-0.005681,0.117614,17.040001,49.435055,-1.787560,7.617661,1.279939,-27.030146,0,0.476838
2015-10-30,-0.001861,0.115428,17.580000,48.428844,-1.770332,7.721606,1.182082,-15.219813,0,0.489299
2015-11-02,0.001229,0.117224,17.879999,38.647577,-1.738055,7.611434,0.809108,-19.172615,0,0.497006


In [ ]:
#| export
class niftyriskdataset(Dataset):
    def __init__(self, df):
        self.x0 = df['target_return'].values.astype(np.float32)
        self.continious_cols = ['realized_vol', 'VIX', "RSI", "skew", "kurtosis", "vol_spike", "RSI_velocity","regime_score"]
        self.context_cont = df[self.continious_cols].values.astype(np.float32)
        self.context_cat = df[['regime']].values.astype('int64')

    def __len__(self):
        return len(self.x0)

    def __getitem__(self, idx):
        return (
            torch.tensor([self.x0[idx]]),
            torch.tensor(self.context_cat[idx][0])+1,
            torch.tensor(self.context_cont[idx])
        )


def prepare_dataloaders(df, batch_size=64, train_size=0.70, val_size=0.20):
    total_size = len(df)
    train_end = int(train_size * total_size)
    val_end = int((train_size + val_size) * total_size)
    train_df_orig = df[:train_end].copy()
    val_df_orig = df[train_end:val_end].copy()
    condition_df_orig = df[val_end:].copy()

    print(f"Data split:")
    print(f"  Train: {len(train_df_orig)} samples ({len(train_df_orig)/total_size:.1%})")
    print(f"  Validation: {len(val_df_orig)} samples ({len(val_df_orig)/total_size:.1%})")
    print(f"  Conditioning: {len(condition_df_orig)} samples ({len(condition_df_orig)/total_size:.1%})")
    scaler_x = StandardScaler()
    scaler_cond = StandardScaler()

    cont_features = ['realized_vol', 'VIX', 'RSI', 'skew', 'kurtosis', 'vol_spike', 'RSI_velocity',"regime_score"]
    scaler_x.fit(train_df_orig[['target_return']])
    scaler_cond.fit(train_df_orig[cont_features])

    train_df_scaled = train_df_orig.copy()
    train_df_scaled[['target_return']] = scaler_x.transform(train_df_orig[['target_return']])
    train_df_scaled[cont_features] = scaler_cond.transform(train_df_orig[cont_features])

    val_df_scaled = val_df_orig.copy()
    val_df_scaled[['target_return']] = scaler_x.transform(val_df_orig[['target_return']])
    val_df_scaled[cont_features] = scaler_cond.transform(val_df_orig[cont_features])
    condition_df = condition_df_orig.copy()
    train_ds = niftyriskdataset(train_df_scaled)
    val_ds = niftyriskdataset(val_df_scaled)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)

    return train_loader, val_loader, condition_df, scaler_x, scaler_cond,val_df_orig



In [ ]:
train_dl, val_dl, condition_df, scaler_x, scaler_cond,val_df = prepare_dataloaders(
    risk_df,
    batch_size=64,
    train_size=0.80,
    val_size=0.10
)

print(f"\nDataLoader batch info:")
print(f"  Train batches: {len(train_dl)}")
print(f"  Validation batches: {len(val_dl)}")
print(f"\nConditioning dataframe shape: {condition_df.shape}")
print(f"Conditioning date range: {condition_df.index[0]} to {condition_df.index[-1]}")

Data split:
  Train: 2037 samples (80.0%)
  Validation: 255 samples (10.0%)
  Conditioning: 255 samples (10.0%)

DataLoader batch info:
  Train batches: 32
  Validation batches: 4

Conditioning dataframe shape: (255, 10)
Conditioning date range: 2025-02-14 00:00:00 to 2026-02-26 00:00:00

DataLoader batch info:
  Train batches: 32
  Validation batches: 4

Conditioning dataframe shape: (255, 10)
Conditioning date range: 2025-02-14 00:00:00 to 2026-02-26 00:00:00
